# IR Vibrational Spectroscopy Widget

This notebook demonstrates the IR vibrational widget for displaying quantum
chemistry calculation results. It covers:

- Static examples using pre-defined frequency data
- A **Psi4** harmonic frequency calculation on H₃O⁺ via PsiAPI
- Interactive and static visualisations of the resulting IR spectrum

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from ir_widget import IRWidget

## Example 1: Water Molecule (H₂O)

Water has 3 normal modes:
- Bending mode (~1595 cm⁻¹)
- Symmetric O–H stretch (~3657 cm⁻¹)
- Asymmetric O–H stretch (~3756 cm⁻¹)

In [ ]:
frequencies = np.array([1595.0, 3657.0, 3756.0])
intensities = np.array([75.0, 20.0, 45.0])

widget_h2o = IRWidget()
widget_h2o.load_data(frequencies, intensities, formula="H2O")
widget_h2o

## Example 2: Benzene (C₆H₆)

A more complex molecule with multiple vibrational modes across the IR spectrum.

In [ ]:
np.random.seed(42)
frequencies_benzene = np.concatenate([
    np.random.uniform(400, 800, 8),
    np.random.uniform(900, 1600, 12),
    np.random.uniform(2800, 3100, 10),
])
frequencies_benzene = np.sort(frequencies_benzene)
intensities_benzene = np.random.exponential(30, 30)

widget_benzene = IRWidget()
widget_benzene.load_data(frequencies_benzene, intensities_benzene, formula="C6H6")
widget_benzene

## Customising the Display

Broadening type, peak width, and wavenumber range can all be adjusted
programmatically after the widget is created:

In [ ]:
widget_benzene.broadening = "gaussian"  # "none", "lorentzian", or "gaussian"
widget_benzene.fwhm = 25.0              # full-width at half-maximum in cm⁻¹
widget_benzene.x_min = 500
widget_benzene.x_max = 3500

## Psi4 Frequency Calculation: H₃O⁺ (Hydronium Ion)

Using PsiAPI directly we can run an HF/6-31G(d,p) geometry optimisation
followed by a harmonic frequency calculation, then load the wavefunction
straight into the widget — no output file or cclib parsing required.

H₃O⁺ has **6 normal modes** (3N − 6 = 3×4 − 6):

| Symmetry | Description | Approx. freq |
|----------|-------------|--------------|
| A₁ | O–H symmetric stretch | ~3530 cm⁻¹ |
| A₁ | Umbrella (inversion) bend | ~1000 cm⁻¹ |
| E  | O–H asymmetric stretch (×2) | ~3640 cm⁻¹ |
| E  | H–O–H scissors bend (×2) | ~1640 cm⁻¹ |

In [ ]:
import psi4

# Clean up any Psi4 scratch files left from previous runs
for f in Path().glob('psi.*.clean'):
    f.unlink()

psi4.set_memory('2 GB')
psi4.set_num_threads(2)
psi4.core.set_output_file('h3o_freq.dat', False)

In [ ]:
# H₃O⁺: charge = +1, singlet; start from a planar C₂ᵥ-ish geometry
h3o = psi4.geometry("""
  1 1
  O  0.0000  0.0000  0.0000
  H  0.9200 -0.5300  0.0000
  H -0.9200 -0.5200  0.0000
  H  0.0000  1.0600  0.0000
""")

psi4.set_options({'reference': 'rhf'})

# Step 1 — optimise to the C₃ᵥ minimum
psi4.optimize('hf/6-31g(d,p)', molecule=h3o)

# Step 2 — harmonic frequencies; return_wfn gives us the wavefunction
energy, wfn = psi4.frequency('hf/6-31g(d,p)', molecule=h3o, return_wfn=True)
print(f'HF/6-31G(d,p) energy: {energy:.6f} Eh')

In [ ]:
# Load frequencies, IR intensities, normal-mode vectors, and atom
# coordinates directly from the wavefunction — no file I/O needed.
widget_h3o = IRWidget()
widget_h3o.load_from_psi4_wfn(wfn)
widget_h3o.broadening = 'lorentzian'
widget_h3o.fwhm = 20.0
widget_h3o

## Inspecting the Results

All vibrational data are available in `widget_h3o.data`, including
Cartesian displacement vectors (`displacements`) and equilibrium atom
coordinates (`atoms`) extracted directly from the wavefunction.

In [ ]:
data = widget_h3o.data

print(f"Molecule : {data['formula']}")
print(f"Modes    : {data['n_modes']}")
if 'atoms' in data:
    print(f"Atoms    : {len(data['atoms'])}")
print()
print(f"{'Mode':>4}  {'Freq (cm\u207b\u00b9)':>13}  {'Intensity (km/mol)':>18}  Displacements")
print("─" * 58)
for mode in data['modes']:
    has_d = 'yes' if 'displacements' in mode else 'no'
    print(f"{mode['mode']:>4}  {mode['frequency']:>13.2f}  {mode['intensity']:>18.4f}  {has_d}")

In [ ]:
# Static Lorentzian-broadened IR spectrum via matplotlib.
# Useful for publication figures and saving to disk.
freqs  = [m['frequency']  for m in data['modes']]
intens = [m['intensity']  for m in data['modes']]

x_range = np.linspace(200, 4200, 5000)
fwhm_plot = 20.0
gamma = fwhm_plot / 2

spectrum = np.zeros_like(x_range)
for f, i in zip(freqs, intens):
    spectrum += i * gamma**2 / ((x_range - f)**2 + gamma**2)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_range, spectrum / spectrum.max(), color='steelblue', linewidth=1.5)

# Conventional IR plot: high wavenumber on the left
ax.invert_xaxis()
ax.set_xlim(4200, 200)
ax.set_xlabel('Wavenumber (cm\u207b\u00b9)', fontsize=12)
ax.set_ylabel('Relative intensity', fontsize=12)
ax.set_title(
    f"IR Spectrum \u2014 {data['formula']}  "
    f"(HF/6-31G(d,p), Lorentzian FWHM\u2009=\u2009{fwhm_plot}\u2009cm\u207b\u00b9)",
    fontsize=12,
)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## Other Ways to Load Data

| Method | When to use |
|--------|-------------|
| `widget.load_from_psi4_wfn(wfn)` | After `psi4.frequency(..., return_wfn=True)` — recommended for Psi4 1.10+ |
| `widget.run_psi4_frequency(geom, method_basis)` | One-shot: runs Psi4 and loads results in a single call |
| `widget.load_file(path)` | Parses a saved output file via cclib (Gaussian, ORCA, NWChem, …) |
| `widget.load_data(freqs, intens)` | Supply arrays directly — useful for testing or literature data |

In [ ]:
# One-shot convenience runner (sets up Psi4, runs, loads automatically)
# widget = IRWidget()
# widget.run_psi4_frequency(
#     geometry='\n  O\n  H 1 0.96\n  H 1 0.96 2 104.5\n',
#     method_basis='hf/sto-3g',
# )

# Load from a saved Psi4 / Gaussian / ORCA output file (via cclib)
# widget = IRWidget(file_path='sample_data/planar_h3o.log')

# Supply arrays directly
# widget = IRWidget()
# widget.load_data([1000, 1640, 3530, 3640], [120, 45, 30, 80], formula='H3O+')